# 3. Model Training - NYC TLC Trip Duration Prediction

This notebook trains multiple regression models using PySpark MLlib to predict trip duration.

**Models:**
1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosted Trees
4. Decision Tree Regressor

In [ ]:
# Import libraries
import yaml
import json
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported!")

In [ ]:
# Initialize Spark with optimized settings
with open('../config/spark_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

spark_config = config['spark']

spark = SparkSession.builder \
    .appName("Model_Training") \
    .master(spark_config['master']) \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

In [ ]:
# Load engineered features
df = spark.read.parquet("../data/processed/nyc_tlc_features")
print(f"Loaded {df.count():,} records with {len(df.columns)} columns")

# Load feature metadata
with open("../data/schemas/feature_metadata.json", "r") as f:
    feature_metadata = json.load(f)

print("\nFeature metadata loaded:")
print(f"Numerical features: {len(feature_metadata['numerical_features'])}")
print(f"Binary features: {len(feature_metadata['binary_features'])}")
print(f"Categorical features: {len(feature_metadata['categorical_features'])}")

## Data Preparation

In [ ]:
# Handle categorical features with StringIndexer
categorical_features = feature_metadata['categorical_features']
indexers = [StringIndexer(inputCol=col, outputCol=col+"_indexed", handleInvalid="keep") 
            for col in categorical_features]

# Assemble all features
numerical_features = feature_metadata['numerical_features']
binary_features = feature_metadata['binary_features']
indexed_categorical = [col+"_indexed" for col in categorical_features]

all_features = numerical_features + binary_features + indexed_categorical

assembler = VectorAssembler(
    inputCols=all_features,
    outputCol="features_raw",
    handleInvalid="skip"
)

# Scale features
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

print(f"Total features: {len(all_features)}")

## Train-Test Split

In [ ]:
# Split data: 70% train, 15% validation, 15% test
train_data, val_data, test_data = df.randomSplit([0.7, 0.15, 0.15], seed=42)

# Cache datasets
train_data.cache()
val_data.cache()
test_data.cache()

print(f"Training set: {train_data.count():,} records")
print(f"Validation set: {val_data.count():,} records")
print(f"Test set: {test_data.count():,} records")

## Model 1: Linear Regression

In [ ]:
print("\n" + "="*50)
print("Training Linear Regression Model")
print("="*50)

start_time = time.time()

# Create Linear Regression model
lr = LinearRegression(
    featuresCol="features",
    labelCol="trip_duration_minutes",
    maxIter=10,
    regParam=0.1,
    elasticNetParam=0.5
)

# Create pipeline
lr_pipeline = Pipeline(stages=indexers + [assembler, scaler, lr])

# Train model
lr_model = lr_pipeline.fit(train_data)

# Make predictions
lr_predictions = lr_model.transform(val_data)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.2f} seconds")

# Evaluate
evaluator_rmse = RegressionEvaluator(
    labelCol="trip_duration_minutes",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="trip_duration_minutes",
    predictionCol="prediction",
    metricName="r2"
)

evaluator_mae = RegressionEvaluator(
    labelCol="trip_duration_minutes",
    predictionCol="prediction",
    metricName="mae"
)

lr_rmse = evaluator_rmse.evaluate(lr_predictions)
lr_r2 = evaluator_r2.evaluate(lr_predictions)
lr_mae = evaluator_mae.evaluate(lr_predictions)

print(f"\nLinear Regression Results:")
print(f"RMSE: {lr_rmse:.4f}")
print(f"R²: {lr_r2:.4f}")
print(f"MAE: {lr_mae:.4f}")

## Model 2: Decision Tree Regressor

In [ ]:
print("\n" + "="*50)
print("Training Decision Tree Regressor")
print("="*50)

start_time = time.time()

# Create Decision Tree model
dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="trip_duration_minutes",
    maxDepth=10,
    maxBins=32
)

# Create pipeline
dt_pipeline = Pipeline(stages=indexers + [assembler, scaler, dt])

# Train model
dt_model = dt_pipeline.fit(train_data)

# Make predictions
dt_predictions = dt_model.transform(val_data)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.2f} seconds")

# Evaluate
dt_rmse = evaluator_rmse.evaluate(dt_predictions)
dt_r2 = evaluator_r2.evaluate(dt_predictions)
dt_mae = evaluator_mae.evaluate(dt_predictions)

print(f"\nDecision Tree Results:")
print(f"RMSE: {dt_rmse:.4f}")
print(f"R²: {dt_r2:.4f}")
print(f"MAE: {dt_mae:.4f}")

## Model 3: Random Forest Regressor

In [ ]:
print("\n" + "="*50)
print("Training Random Forest Regressor")
print("="*50)

start_time = time.time()

# Create Random Forest model
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="trip_duration_minutes",
    numTrees=50,
    maxDepth=10,
    maxBins=32,
    seed=42
)

# Create pipeline
rf_pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])

# Train model
rf_model = rf_pipeline.fit(train_data)

# Make predictions
rf_predictions = rf_model.transform(val_data)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.2f} seconds")

# Evaluate
rf_rmse = evaluator_rmse.evaluate(rf_predictions)
rf_r2 = evaluator_r2.evaluate(rf_predictions)
rf_mae = evaluator_mae.evaluate(rf_predictions)

print(f"\nRandom Forest Results:")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R²: {rf_r2:.4f}")
print(f"MAE: {rf_mae:.4f}")

# Feature importance
rf_model_stage = rf_model.stages[-1]
feature_importance = rf_model_stage.featureImportances
print(f"\nTop 10 Important Features:")
importance_list = [(all_features[i], float(feature_importance[i])) 
                   for i in range(len(all_features))]
importance_list.sort(key=lambda x: x[1], reverse=True)
for feat, imp in importance_list[:10]:
    print(f"{feat}: {imp:.4f}")

## Model 4: Gradient Boosted Trees

In [ ]:
print("\n" + "="*50)
print("Training Gradient Boosted Trees")
print("="*50)

start_time = time.time()

# Create GBT model
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="trip_duration_minutes",
    maxIter=20,
    maxDepth=5,
    maxBins=32,
    seed=42
)

# Create pipeline
gbt_pipeline = Pipeline(stages=indexers + [assembler, scaler, gbt])

# Train model
gbt_model = gbt_pipeline.fit(train_data)

# Make predictions
gbt_predictions = gbt_model.transform(val_data)

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.2f} seconds")

# Evaluate
gbt_rmse = evaluator_rmse.evaluate(gbt_predictions)
gbt_r2 = evaluator_r2.evaluate(gbt_predictions)
gbt_mae = evaluator_mae.evaluate(gbt_predictions)

print(f"\nGradient Boosted Trees Results:")
print(f"RMSE: {gbt_rmse:.4f}")
print(f"R²: {gbt_r2:.4f}")
print(f"MAE: {gbt_mae:.4f}")

# Feature importance
gbt_model_stage = gbt_model.stages[-1]
feature_importance = gbt_model_stage.featureImportances
print(f"\nTop 10 Important Features:")
importance_list = [(all_features[i], float(feature_importance[i])) 
                   for i in range(len(all_features))]
importance_list.sort(key=lambda x: x[1], reverse=True)
for feat, imp in importance_list[:10]:
    print(f"{feat}: {imp:.4f}")

## Model Comparison

In [ ]:
# Create comparison dataframe
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Random Forest', 'Gradient Boosted Trees'],
    'RMSE': [lr_rmse, dt_rmse, rf_rmse, gbt_rmse],
    'R²': [lr_r2, dt_r2, rf_r2, gbt_r2],
    'MAE': [lr_mae, dt_mae, rf_mae, gbt_mae]
})

print("\n" + "="*70)
print("MODEL COMPARISON (Validation Set)")
print("="*70)
print(results.to_string(index=False))
print("="*70)

# Find best model
best_model_idx = results['RMSE'].idxmin()
best_model_name = results.loc[best_model_idx, 'Model']
print(f"\nBest Model: {best_model_name}")

## Hyperparameter Tuning for Best Model

In [ ]:
# Tune Random Forest (typically performs well)
print("\nPerforming hyperparameter tuning for Random Forest...")

rf_tuned = RandomForestRegressor(
    featuresCol="features",
    labelCol="trip_duration_minutes",
    seed=42
)

# Create parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(rf_tuned.numTrees, [30, 50]) \
    .addGrid(rf_tuned.maxDepth, [8, 10]) \
    .build()

# Create pipeline
rf_tuned_pipeline = Pipeline(stages=indexers + [assembler, scaler, rf_tuned])

# Cross validator
crossval = CrossValidator(
    estimator=rf_tuned_pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_rmse,
    numFolds=3,
    parallelism=2
)

# Train
cv_model = crossval.fit(train_data)

# Best model predictions
best_predictions = cv_model.transform(val_data)

# Evaluate
best_rmse = evaluator_rmse.evaluate(best_predictions)
best_r2 = evaluator_r2.evaluate(best_predictions)
best_mae = evaluator_mae.evaluate(best_predictions)

print(f"\nTuned Random Forest Results:")
print(f"RMSE: {best_rmse:.4f}")
print(f"R²: {best_r2:.4f}")
print(f"MAE: {best_mae:.4f}")

## Save Models

In [ ]:
# Save all models
lr_model.write().overwrite().save("../models/linear_regression")
dt_model.write().overwrite().save("../models/decision_tree")
rf_model.write().overwrite().save("../models/random_forest")
gbt_model.write().overwrite().save("../models/gradient_boosted_trees")
cv_model.write().overwrite().save("../models/random_forest_tuned")

print("All models saved successfully!")

# Save results
results.to_csv("../data/processed/model_comparison.csv", index=False)
print("Results saved to CSV")

In [ ]:
print("\nModel training complete!")